In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fuzzy Adaptive Quadrotor PID — Demo\n",
    "\n",
    "This notebook demonstrates the complete pipeline:\n",
    "1. Build a quadrotor model\n",
    "2. Apply scaling laws\n",
    "3. Apply fuzzy adaptive scaling\n",
    "4. Simulate and compare"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys, os\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.dynamics.quadrotor_model import QuadrotorDynamics, QuadrotorParameters\n",
    "from src.scaling.scaling_laws import ScalingLaw, generate_variants\n",
    "from src.control.fuzzy_adaptive import FuzzyAdaptiveScaler\n",
    "from src.simulation.scenarios import get_default_scenarios\n",
    "from src.simulation.simulator import Simulator"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Baseline gains and quadrotor variants"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "BASELINE_GAINS = np.array([\n",
    "    12.5, 1.25, 3.125,\n",
    "    6.25, 0.075, 1.25,\n",
    "    6.25, 0.075, 1.25,\n",
    "    5.0, 0.05, 1.0,\n",
    "])\n",
    "\n",
    "variants = generate_variants()\n",
    "for f, v in variants.items():\n",
    "    print(f'{f}×  mass={v[\"mass\"]:.3f} kg  arm={v[\"arm_length\"]:.3f} m')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Scaling law transfer to a 5× quadrotor"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "scaler = ScalingLaw(1.0, 0.25)\n",
    "v5 = variants[5.0]\n",
    "scaled_gains = scaler.scale_gains(BASELINE_GAINS, v5['mass'], v5['arm_length'])\n",
    "print('Scaled gains:')\n",
    "print(scaled_gains)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Fuzzy adaptive correction under heavy payload + wind"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fuzzy = FuzzyAdaptiveScaler()\n",
    "adaptive_gains = fuzzy.apply_to_gains(\n",
    "    scaled_gains,\n",
    "    payload_variation=30.0,\n",
    "    turbulence_level=0.7,\n",
    "    battery_state=0.4,\n",
    "    error_dynamics=0.5,\n",
    ")\n",
    "print('Adaptive gains:')\n",
    "print(adaptive_gains)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Simulate and compare"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "params = QuadrotorParameters(\n",
    "    mass=v5['mass'], arm_length=v5['arm_length'],\n",
    "    Ixx=v5['Ixx'], Iyy=v5['Iyy'], Izz=v5['Izz'],\n",
    ")\n",
    "dyn = QuadrotorDynamics(params)\n",
    "sim = Simulator(dyn)\n",
    "\n",
    "scenario = get_default_scenarios()[0]\n",
    "\n",
    "res_static = sim.simulate(scaled_gains, scenario.reference)\n",
    "res_fuzzy = sim.simulate(adaptive_gains, scenario.reference)\n",
    "\n",
    "print(f'Static  fitness: {res_static.fitness:.4f}')\n",
    "print(f'Fuzzy   fitness: {res_fuzzy.fitness:.4f}')\n",
    "\n",
    "plt.figure(figsize=(10, 4))\n",
    "plt.plot(res_static.t, res_static.X[2, :], label='Static scaled')\n",
    "plt.plot(res_fuzzy.t, res_fuzzy.X[2, :], label='Fuzzy adaptive')\n",
    "plt.axhline(scenario.reference[0], ls='--', color='gray', label='Reference')\n",
    "plt.xlabel('Time (s)')\n",
    "plt.ylabel('Altitude z (m)')\n",
    "plt.title('Altitude tracking — 5× quadrotor')\n",
    "plt.legend()\n",
    "plt.grid(alpha=0.3)\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}